# Weighted competing pricing from Parquet across elasticity

This notebook reads two competitor predictions, exposure, and observed loss from an arbitrary Parquet file. It then computes loss-ratio and profit curves for elasticity values from 0 to 10 using deterministic probability weights.

The input must contain one row per policy or risk. Column names are configurable below.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## Input and pricing settings

Set `PARQUET_PATH` and the four column names to match the input file. `ID_COLUMN` is optional and is only used for the preview. The elasticity grid includes 0 and 10.

In [ ]:
PARQUET_PATH = Path("data/03_predictions/competing_pricing.parquet")
ID_COLUMN = None
PREDICTION_A_COLUMN = "prediction_a"
PREDICTION_B_COLUMN = "prediction_b"
EXPOSURE_COLUMN = "exposure"
LOSS_COLUMN = "loss"

LOADING_A = 1.0
LOADING_B = 1.0
ALIGN_PREDICTIONS = True
ELASTICITY_MIN = 0.0
ELASTICITY_MAX = 10.0
ELASTICITY_STEP = 0.1

if not PARQUET_PATH.is_file():
    raise FileNotFoundError(f"Parquet file not found: {PARQUET_PATH}")
if ELASTICITY_MIN < 0 or ELASTICITY_MAX < ELASTICITY_MIN:
    raise ValueError("The elasticity range must satisfy 0 <= min <= max.")
if ELASTICITY_STEP <= 0:
    raise ValueError("ELASTICITY_STEP must be positive.")
if LOADING_A <= 0 or LOADING_B <= 0:
    raise ValueError("Commercial loadings must be positive.")

elasticities = np.arange(
    ELASTICITY_MIN, ELASTICITY_MAX + ELASTICITY_STEP / 2, ELASTICITY_STEP
)
elasticities = elasticities[elasticities <= ELASTICITY_MAX + 1e-12]

In [ ]:
required_columns = [
    PREDICTION_A_COLUMN,
    PREDICTION_B_COLUMN,
    EXPOSURE_COLUMN,
    LOSS_COLUMN,
]
data = pd.read_parquet(PARQUET_PATH)
missing_columns = sorted(set(required_columns) - set(data.columns))
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

portfolio = data[required_columns].copy()
portfolio.columns = ["prediction_a", "prediction_b", "exposure", "loss"]
if portfolio.empty:
    raise ValueError("The input Parquet file contains no rows.")
if portfolio.isna().any().any():
    raise ValueError("Required input columns must not contain missing values.")
if not np.isfinite(portfolio.to_numpy(dtype=float)).all():
    raise ValueError("Required input columns must contain finite numeric values.")
if (portfolio[["prediction_a", "prediction_b", "exposure"]] <= 0).any().any():
    raise ValueError("Predictions and exposure must be positive.")
if (portfolio["loss"] < 0).any():
    raise ValueError("Loss must be non-negative.")

print(f"Rows loaded: {len(portfolio):,}")
portfolio.head()

## Align predictions and create loaded quotes

For each competitor $j$, the optional alignment factor is

$$c_j = \frac{\sum_i L_i}{\sum_i \widehat{R}_{ij}E_i}.$$

After alignment, each prediction has global ELR equal to one before the commercial loading.

In [ ]:
def align_prediction(prediction, exposure, loss):
    factor = loss.sum() / (prediction * exposure).sum()
    return prediction * factor, factor


if ALIGN_PREDICTIONS:
    portfolio["prediction_a"], factor_a = align_prediction(
        portfolio["prediction_a"], portfolio["exposure"], portfolio["loss"]
    )
    portfolio["prediction_b"], factor_b = align_prediction(
        portfolio["prediction_b"], portfolio["exposure"], portfolio["loss"]
    )
else:
    factor_a = factor_b = 1.0

portfolio["quote_a"] = LOADING_A * portfolio["prediction_a"]
portfolio["quote_b"] = LOADING_B * portfolio["prediction_b"]
print(f"ELR alignment factors: A={factor_a:.6g}, B={factor_b:.6g}")
portfolio[["quote_a", "quote_b"]].head()

## Compute LR and profit curves

The won-business totals use the choice probability as a weight, while lost-business totals use its complement. `profit_rate` is profit on won business divided by total offered exposure.

In [ ]:
def safe_ratio(numerator, denominator):
    return numerator / denominator if denominator > 0 else np.nan


def weighted_kpis(portfolio, elasticity, competitor):
    quote = portfolio[f"quote_{competitor}"]
    other = "b" if competitor == "a" else "a"
    other_quote = portfolio[f"quote_{other}"]
    weight = 1 / (1 + (quote / other_quote) ** elasticity)
    premium = quote * portfolio["exposure"]
    loss = portfolio["loss"]
    won_loss = (weight * loss).sum()
    won_premium = (weight * premium).sum()
    lost_loss = ((1 - weight) * loss).sum()
    lost_premium = ((1 - weight) * premium).sum()
    profit = won_premium - won_loss
    return {
        "elasticity": elasticity,
        "competitor": competitor.upper(),
        "expected_share_won": weight.mean(),
        "lr_win": safe_ratio(won_loss, won_premium),
        "lr_lose": safe_ratio(lost_loss, lost_premium),
        "profit": profit,
        "profit_rate": safe_ratio(profit, portfolio["exposure"].sum()),
        "expected_won_premium": won_premium,
        "expected_won_loss": won_loss,
    }

results = pd.DataFrame(
    [
        weighted_kpis(portfolio, elasticity, competitor)
        for elasticity in elasticities
        for competitor in ("a", "b")
    ]
)
results.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True)
for competitor, group in results.groupby("competitor"):
    axes[0].plot(group["elasticity"], group["lr_win"], label=f"{competitor} — won")
    axes[0].plot(
        group["elasticity"], group["lr_lose"], linestyle="--", label=f"{competitor} — lost"
    )
    axes[1].plot(group["elasticity"], group["profit_rate"], label=competitor)
axes[0].set_title("Loss ratio across elasticity")
axes[0].set_xlabel("Elasticity")
axes[0].set_ylabel("Loss ratio")
axes[0].grid(alpha=0.25)
axes[0].legend()
axes[1].set_title("Profit rate across elasticity")
axes[1].set_xlabel("Elasticity")
axes[1].set_ylabel("Profit / offered exposure")
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].grid(alpha=0.25)
axes[1].legend(title="Competitor")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for competitor, group in results.groupby("competitor"):
    ax.plot(group["elasticity"], group["profit"], label=competitor)
ax.set_title("Expected profit across elasticity")
ax.set_xlabel("Elasticity")
ax.set_ylabel("Expected profit")
ax.axhline(0, color="black", linewidth=0.8)
ax.grid(alpha=0.25)
ax.legend(title="Competitor")
plt.tight_layout()
plt.show()